In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DE_FinalWithGenerated.csv')

# Convert time to datetime format
df['time'] = pd.to_datetime(df['time'], format='ISO8601')

# List of categorical columns to encode
categorical_columns = ['road_type', 'weather_condition', 'road_condition', 'time_of_day']

# Initialize the OneHotEncoder
encoder = OneHotEncoder(sparse=False, drop='first')

# Fit and transform the categorical columns
encoded_columns = encoder.fit_transform(df[categorical_columns])

# Get the new column names
encoded_column_names = encoder.get_feature_names_out(categorical_columns)

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded_columns, columns=encoded_column_names)

# Concatenate the original DataFrame with the encoded DataFrame
df = pd.concat([df, encoded_df], axis=1)

# Drop the original categorical columns
df.drop(columns=categorical_columns, inplace=True)

# Splitting by country and sorting by collection car and time within each country
grouped_by_country = df.groupby('country_code')


In [ ]:
def train_and_evaluate_model(X_train, y_train, X_test, y_test, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    medae = median_absolute_error(y_test, predictions)
    return mse, r2, mae, medae

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(),
    'XGBoost': xgb.XGBRegressor(objective='reg:squarederror'),
    'SVR': SVR(),
    'DecisionTree': DecisionTreeRegressor(),
    'GradientBoosting': GradientBoostingRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'KNN': KNeighborsRegressor(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'BayesianRidge': BayesianRidge(),
    'NeuralNetwork': MLPRegressor()
}

In [ ]:
results = {}

for country, group in grouped_by_country:
    group = group.sort_values(by=['collection_car', 'time'])

    # Check if the group has enough samples
    if len(group) < 5:  # you can adjust the threshold as needed
        continue

    # Define features and target
    X = group.drop(columns=['frame_id', 'country_code', 'collection_car', 'time', 'average_speed_kmh'])
    y = group['average_speed_kmh']

    # Splitting the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

    results[country] = {}

    for model_name, model in models.items():
        mse, r2, mae, medae = train_and_evaluate_model(X_train, y_train, X_test, y_test, model)
        results[country][model_name] = {'MSE': mse, 'R2': r2, 'MAE': mae, 'MedAE': medae}

# Displaying results
for key, value in results.items():
    print(f'Country: {key}')
    for model_name, metrics in value.items():
        print(f'  Model: {model_name}, R2: {metrics["R2"]:.4f}, MAE: {metrics["MAE"]:.4f}, MedAE: {metrics["MedAE"]:.4f}')
    print('\n')

Country: DE
  Model: LinearRegression, R2: 0.6708, MAE: 11.1274, MedAE: 8.6503
  Model: RandomForest, R2: 0.6429, MAE: 11.2004, MedAE: 8.6000
  Model: XGBoost, R2: 0.6426, MAE: 11.3913, MedAE: 8.7252
  Model: SVR, R2: 0.2170, MAE: 15.0656, MedAE: 9.4541
  Model: DecisionTree, R2: 0.3845, MAE: 15.1385, MedAE: 12.0000
  Model: GradientBoosting, R2: 0.6833, MAE: 10.7668, MedAE: 8.4153
  Model: AdaBoost, R2: 0.6872, MAE: 10.8655, MedAE: 8.5989
  Model: KNN, R2: 0.6172, MAE: 11.9528, MedAE: 9.4000
  Model: Ridge, R2: 0.6706, MAE: 11.1294, MedAE: 8.6370
  Model: Lasso, R2: 0.1981, MAE: 15.3634, MedAE: 9.9648
  Model: ElasticNet, R2: 0.2013, MAE: 15.4048, MedAE: 9.9536
  Model: BayesianRidge, R2: 0.6699, MAE: 11.1386, MedAE: 8.6331
  Model: NeuralNetwork, R2: 0.6782, MAE: 10.9334, MedAE: 8.7730




In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Final_Dataset_DE.csv')

# Convert time to datetime format
df['time'] = pd.to_datetime(df['time'], format='ISO8601')

# Extract temporal features from time column
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['day_of_month'] = df['time'].dt.day
df['month'] = df['time'].dt.month
df['year'] = df['time'].dt.year

# List of categorical columns to encode
categorical_columns = ['road_type', 'weather_condition', 'road_condition', 'time_of_day']

# Initialize the OneHotEncoder
encoder = OneHotEncoder(sparse=False, drop='first')

# Fit and transform the categorical columns
encoded_columns = encoder.fit_transform(df[categorical_columns])

# Get the new column names
encoded_column_names = encoder.get_feature_names_out(categorical_columns)

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded_columns, columns=encoded_column_names)

# Concatenate the original DataFrame with the encoded DataFrame
df = pd.concat([df, encoded_df], axis=1)

# Drop the original categorical columns
df.drop(columns=categorical_columns, inplace=True)

# Splitting by country and sorting by collection car and time within each country
grouped_by_country = df.groupby('country_code')

def train_and_evaluate_model(X_train, y_train, X_test, y_test, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    medae = median_absolute_error(y_test, predictions)
    return mse, r2, mae, medae

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(),
    'XGBoost': xgb.XGBRegressor(objective='reg:squarederror'),
    'GradientBoosting': GradientBoostingRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'Ridge': Ridge(),
    'BayesianRidge': BayesianRidge(),
    'NeuralNetwork': MLPRegressor()
}

results = {}

for country, group in grouped_by_country:
    group = group.sort_values(by=['collection_car', 'time'])

    # Check if the group has enough samples
    if len(group) < 5:  # you can adjust the threshold as needed
        continue

    # Define features and target
    X = group.drop(columns=['frame_id', 'country_code', 'collection_car', 'time', 'average_speed_kmh'])
    y = group['average_speed_kmh']

    # Splitting the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

    results[country] = {}

    for model_name, model in models.items():
        mse, r2, mae, medae = train_and_evaluate_model(X_train, y_train, X_test, y_test, model)
        results[country][model_name] = {'MSE': mse, 'R2': r2, 'MAE': mae, 'MedAE': medae}

# Displaying results
for key, value in results.items():
    print(f'Country: {key}')
    for model_name, metrics in value.items():
        print(f'  Model: {model_name}, R2: {metrics["R2"]:.4f}, MAE: {metrics["MAE"]:.4f}, MedAE: {metrics["MedAE"]:.4f}')
    print('\n')


Country: DE
  Model: LinearRegression, R2: 0.6959, MAE: 10.6073, MedAE: 8.2670
  Model: RandomForest, R2: 0.7151, MAE: 10.2459, MedAE: 8.0850
  Model: XGBoost, R2: 0.6370, MAE: 11.2558, MedAE: 8.2818
  Model: GradientBoosting, R2: 0.7142, MAE: 10.1476, MedAE: 7.5462
  Model: AdaBoost, R2: 0.7129, MAE: 10.5560, MedAE: 8.6961
  Model: Ridge, R2: 0.6958, MAE: 10.6092, MedAE: 8.2454
  Model: BayesianRidge, R2: 0.6948, MAE: 10.6214, MedAE: 8.2219
  Model: NeuralNetwork, R2: 0.6668, MAE: 10.8998, MedAE: 8.1364


